# Teks — Tuning Hyperparameter (TF-IDF + model linear, Optuna)
**Kapan dipakai:** baseline TF-IDF sudah jalan dan ingin skor naik tanpa ganti metode.

**Yang dicari:** pengaturan TF-IDF (n-gram kata/huruf, min_df, max_features) dan kekuatan
model (C). Tiap kombinasi dinilai dengan 5-fold CV di data latih; data validasi hanya dipakai
di akhir.

**Data demo:** SmSA (sentimen 3 kelas).

In [1]:
import optuna
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.svm import LinearSVC

optuna.logging.set_verbosity(optuna.logging.WARNING)

## Konfigurasi

In [2]:
DATA_FOLDER = "../../../data_latihan/text/sentiment"   # GANTI saat lomba
TRAIN_CSV = f"{DATA_FOLDER}/smsa_train.csv"
TEST_CSV = f"{DATA_FOLDER}/smsa_test.csv"

TEXT_COL = "text"
TARGET = "label"
N_TRIALS = 30
SEED = 42

## Baca data

In [3]:
train = pd.read_csv(TRAIN_CSV)
test = pd.read_csv(TEST_CSV)
train[TEXT_COL] = train[TEXT_COL].fillna("")
test[TEXT_COL] = test[TEXT_COL].fillna("")
print("train:", train.shape, "| test:", test.shape)
print(train[TARGET].value_counts())

train: (5148, 3) | test: (1260, 2)
label
negative    2000
positive    2000
neutral     1148
Name: count, dtype: int64


## Split train / validasi

In [4]:
X_train, X_val, y_train, y_val = train_test_split(
    train[TEXT_COL], train[TARGET], test_size=0.2, stratify=train[TARGET], random_state=SEED
)

## Baseline (pengaturan bawaan) sebagai pembanding

In [5]:
baseline = make_pipeline(TfidfVectorizer(), LogisticRegression(max_iter=2000))
baseline_scores = cross_val_score(baseline, X_train, y_train, cv=5, scoring="f1_macro")
print("CV F1 baseline:", round(baseline_scores.mean(), 4))

CV F1 baseline: 0.836


## Ruang pencarian + Optuna
`analyzer="char_wb"` = potongan huruf (tahan salah ketik/singkatan); `"word"` = kata utuh.
Ganti model: beri `#` pada `return` yang aktif, hapus `#` pada `return` lain.

In [6]:
def make_pipe(trial):
    analyzer = trial.suggest_categorical("analyzer", ["word", "char_wb"])
    if analyzer == "word":
        ngram_max = trial.suggest_int("ngram_max_kata", 1, 3)
    else:
        ngram_max = trial.suggest_int("ngram_max_huruf", 3, 6)
    vectorizer = TfidfVectorizer(
        analyzer=analyzer,
        ngram_range=(1, ngram_max),
        min_df=trial.suggest_int("min_df", 1, 5),
        max_features=trial.suggest_categorical("max_features", [20000, 50000, 100000]),
        sublinear_tf=trial.suggest_categorical("sublinear_tf", [True, False]),
    )
    strength = trial.suggest_float("C", 0.01, 100, log=True)
    return make_pipeline(vectorizer, LogisticRegression(C=strength, max_iter=3000))
    # return make_pipeline(vectorizer, LinearSVC(C=strength))


def objective(trial):
    scores = cross_val_score(make_pipe(trial), X_train, y_train, cv=5, scoring="f1_macro")
    return scores.mean()


study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=N_TRIALS)
print("CV F1 terbaik:", round(study.best_value, 4), "| baseline:", round(baseline_scores.mean(), 4))
print(study.best_params)

CV F1 terbaik: 0.8594 | baseline: 0.836
{'analyzer': 'word', 'ngram_max_kata': 3, 'min_df': 2, 'max_features': 20000, 'sublinear_tf': True, 'C': 22.76054768177576}


## 10 percobaan terbaik (lihat pola: analyzer apa yang menang?)

In [7]:
history = study.trials_dataframe()
history.sort_values("value", ascending=False).head(10)

,number,value,datetime_start,datetime_complete,duration,params_C,params_analyzer,params_max_features,params_min_df,params_ngram_max_huruf,params_ngram_max_kata,params_sublinear_tf,system_attrs_tpe:relative_params:0,state
25,25,0.859409,2026-10-19 20:17:30.449826,2026-10-19 20:17:35.625166,0 days 00:00:05.175340,22.760548,word,20000,2,NaN,3.0,True,"{""C"": 22.76054768177576, ""analyzer"": ""word"", ""...",COMPLETE
27,27,0.858665,2026-10-19 20:17:39.571340,2026-10-19 20:17:45.114522,0 days 00:00:05.543182,22.993803,word,20000,2,NaN,3.0,True,"{""C"": 22.99380330334943, ""analyzer"": ""word"", ""...",COMPLETE
28,28,0.858487,2026-10-19 20:17:45.114576,2026-10-19 20:17:50.173554,0 days 00:00:05.058978,21.375307,word,20000,2,NaN,3.0,True,"{""C"": 21.37530707687583, ""analyzer"": ""word"", ""...",COMPLETE
11,11,0.857028,2026-10-19 20:15:58.168901,2026-10-19 20:16:03.299694,0 days 00:00:05.130793,92.992711,word,50000,2,NaN,3.0,True,"{""C"": 92.99271067317383, ""analyzer"": ""word"", ""...",COMPLETE
6,6,0.856332,2026-10-19 20:15:37.254427,2026-10-19 20:15:42.177704,0 days 00:00:04.923277,88.623265,word,50000,2,NaN,3.0,True,NaN,COMPLETE
13,13,0.855902,2026-10-19 20:16:10.347366,2026-10-19 20:16:13.854277,0 days 00:00:03.506911,94.835942,word,50000,2,NaN,2.0,True,"{""C"": 94.83594187035324, ""analyzer"": ""word"", ""...",COMPLETE
14,14,0.855713,2026-10-19 20:16:13.854330,2026-10-19 20:16:17.717848,0 days 00:00:03.863518,85.262221,word,20000,2,NaN,2.0,True,"{""C"": 85.26222101304253, ""analyzer"": ""word"", ""...",COMPLETE
17,17,0.855454,2026-10-19 20:16:40.155738,2026-10-19 20:16:44.782493,0 days 00:00:04.626755,16.644729,word,50000,2,NaN,3.0,False,"{""C"": 16.644728775071908, ""analyzer"": ""word"", ...",COMPLETE
26,26,0.855366,2026-10-19 20:17:35.625219,2026-10-19 20:17:39.571285,0 days 00:00:03.946066,5.773254,word,20000,3,NaN,3.0,True,"{""C"": 5.773253652314149, ""analyzer"": ""word"", ""...",COMPLETE
24,24,0.855026,2026-10-19 20:17:26.453871,2026-10-19 20:17:30.449774,0 days 00:00:03.995903,3.559816,word,50000,3,NaN,3.0,True,"{""C"": 3.559815559373671, ""analyzer"": ""word"", ""...",COMPLETE


## Evaluasi di validasi (pengaturan terbaik)

In [8]:
pipe = make_pipe(study.best_trial)
pipe.fit(X_train, y_train)
val_predictions = pipe.predict(X_val)
print("F1 macro validasi:", round(f1_score(y_val, val_predictions, average="macro"), 4))
print(classification_report(y_val, val_predictions))

F1 macro validasi: 0.8621
              precision    recall  f1-score   support

    negative       0.85      0.85      0.85       400
     neutral       0.86      0.83      0.85       230
    positive       0.88      0.90      0.89       400

    accuracy                           0.87      1030
   macro avg       0.86      0.86      0.86      1030
weighted avg       0.86      0.87      0.86      1030



## Latih ulang di semua data + submission

In [9]:
pipe.fit(train[TEXT_COL], train[TARGET])
predictions = pipe.predict(test[TEXT_COL])

submission = pd.DataFrame()
submission["id"] = test["id"]
submission["label"] = predictions
submission.to_csv("submission.csv", index=False)
submission.head()

,id,label
0,100001,negative
1,100002,negative
2,100003,positive
3,100004,positive
4,100005,negative
